In [1]:
import pandas as pd
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [3]:
df_train = pd.read_csv('GermanTrafficSign/Train.csv')
df_test = pd.read_csv('GermanTrafficSign/Test.csv')

In [4]:
df_train

,Width,Height,Roi.X1,Roi.Y1,Roi.X2,Roi.Y2,ClassId,Path
0,27,26,5,5,22,20,20,Train/20/00020_00000_00000.png
1,28,27,5,6,23,22,20,Train/20/00020_00000_00001.png
2,29,26,6,5,24,21,20,Train/20/00020_00000_00002.png
3,28,27,5,6,23,22,20,Train/20/00020_00000_00003.png
4,28,26,5,5,23,21,20,Train/20/00020_00000_00004.png
...,...,...,...,...,...,...,...,...
39204,52,56,5,6,47,51,42,Train/42/00042_00007_00025.png
39205,56,58,5,5,51,53,42,Train/42/00042_00007_00026.png
39206,58,62,5,6,53,57,42,Train/42/00042_00007_00027.png
39207,63,69,5,7,58,63,42,Train/42/00042_00007_00028.png


# Klasa przygotowywująca dane do treningu modeli.
# 1. __init__ - zmienne potrzebne do zainicjowania danych.
# 2. __len__ - zwraca wielkość zbioru (ilość próbek).
# 3. __getitem__ - pobieranie próbek wraz z klasą o określonym indeksie oraz transformacja zdjęć za pomocą określonych parametrów.

In [5]:
# https://www.geeksforgeeks.org/deep-learning/how-do-you-use-pytorchs-dataset-and-dataloader-classes-for-custom-data/
class CustomDataset(Dataset):
    def __init__(self, data, labels, transform):
        self.data = data
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img = Image.open(self.data[idx])
        sample = transform(img)
        label = self.labels[idx]
        return sample, label

# Przygotowanie danych (zdjęć i przynależności znaku do klasy), transformacja zdjęć do takich samych rozmiarów, zmiana na tensory oraz załadowanie danych do treningu i walidacji.

In [6]:
# https://docs.pytorch.org/docs/2.14/data.html
data_train = 'GermanTrafficSign/' + df_train['Path']
labels_train = df_train['ClassId'].tolist()

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

train = CustomDataset(data_train, labels_train, transform=transform)
train, valid = random_split(train, [0.9, 0.1], generator=torch.Generator().manual_seed(42))

trainloader = DataLoader(train, batch_size=32, shuffle=True)
validloader = DataLoader(valid, batch_size=32, shuffle=True)

# Schemat CNN potrzebny do treningu modelu.
# 1. Conv2d - wyodrębnia za pomocą różnych filtrów ze zdjęcia ważne szczegóły, które mogą decydować o przynależności do danej klasy.
# 2. MaxPool2d - wybiera największą wartość wsród pikseli (filter 2x2), a reszta jest pomijana, dzięki czemu zmniejsza rozmiar zdjęcia.
# 3. relu - zamienia wszystkie wartości ujemne na 0, dzięki czemu wprowadza nieliniowość, która pozwala na uczenie złożonych wzorców.
# 4. Linear - przetwarza wektor cech redukując je do liczby odpowiadającej ilości klas, co pozwala na ustaleninie przynależności do danej klasy.

In [7]:
# https://www.geeksforgeeks.org/deep-learning/building-a-convolutional-neural-network-using-pytorch/
class GermanTrafficSignCNN(nn.Module):
    def __init__(self):
        super(GermanTrafficSignCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, 1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, 1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(128 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 43)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 128 * 8 * 8)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Przygotowanie modelu do treningu.
# 1. CrossEntropyLoss() - funkcja straty, która mierzy jak bardzo przewidywania modelu różnią się od rzeczywistych etykiet. Model zwraca logity, czyli surowe wyniki, zaś funkcja straty zamienia je na prawdopodobieństwa za pomocą softmax dla każdej klasy.
# 2. optim.Adam() - aktualizuje wagi podczas treningu, aby zmniejszyć funkcję straty i tym samym ulepszyć działanie modelu. Optymalizator ma ustawiony learning rate, aby umożliwić skuteczny trening modelu aktualizując ich wagi. Jeśli learning rate jest za duży to model może nigdy nie nauczyć się rozpoznawać poprawnie wzorce i rozpoznawać odpowiednią klasę, a jeśli learning rate byłby zbyt mały to trening trwałby zbyt długo.

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GermanTrafficSignCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Trening modelu CNN oraz walidacja podczas treningu. Najlepszy model zapisywany jest do pliku .pth podczas treningu, gdy obecna strata jest mniejsza niż poprzednia najlepsza, co oznacza, że model działa lepiej. Model osiągnął najlepsze wyniki przy epoce 35.

In [9]:
# https://www.geeksforgeeks.org/deep-learning/save-and-load-models-in-pytorch/
# https://www.geeksforgeeks.org/machine-learning/training-neural-networks-with-validation-using-pytorch/
num_epoch = 100
min_valid_loss = np.inf

for epoch in range(num_epoch):
    train_loss = 0
    model.train()
    for images, labels in trainloader:
        # Trening
        if torch.cuda.is_available():
            images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        target = model(images)
        loss = criterion(target, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Walidacja
    valid_loss = 0
    correct = 0
    total = 0
    model.eval()
    for images, labels in validloader:
        if torch.cuda.is_available():
            images, labels = images.to(device), labels.to(device)

        target = model(images)
        loss = criterion(target, labels)
        valid_loss = loss.item() * images.size(0)
        _, predicted = torch.max(target.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    log_message = f'Epoch [{epoch+1}/{num_epoch}], Loss: {train_loss/len(trainloader):.4f}, '
    log_message += f'Validation: {valid_loss/len(validloader)}, Validation Accuracy: {100 * correct/total:.2f}%.'
    print(log_message)
    with open('log_messages.txt', 'a', encoding='utf-8') as f:
        f.write(log_message + '\n')

    if min_valid_loss > valid_loss:
        log_message2 = f'Epoch [{epoch+1}/{num_epoch}], Validation Loss Decreased: {min_valid_loss:.6f} ---> {valid_loss:.6f}, Model Saved.'
        print(log_message2)
        with open('log_messages2.txt', 'a', encoding='utf-8') as f2:
            f2.write(log_message2 + '\n')
        min_valid_loss = valid_loss
        torch.save(model.state_dict(), 'CNN_GermanTrafficSign_best_model.pth')

Epoch [1/100], Loss: 1.2599, Validation: 0.00471060208188809, Validation Accuracy: 92.98%.
Epoch [1/100], Validation Loss Decreased: inf ---> 0.579404, Model Saved.
Epoch [2/100], Loss: 0.1413, Validation: 0.014302211079171033, Validation Accuracy: 97.22%.
Epoch [3/100], Loss: 0.0650, Validation: 0.010932683944702148, Validation Accuracy: 98.21%.
Epoch [4/100], Loss: 0.0431, Validation: 0.00021072309009912538, Validation Accuracy: 99.06%.
Epoch [4/100], Validation Loss Decreased: 0.579404 ---> 0.025919, Model Saved.
Epoch [5/100], Loss: 0.0336, Validation: 0.042048888477852674, Validation Accuracy: 98.55%.
Epoch [6/100], Loss: 0.0280, Validation: 2.3094587012883126e-05, Validation Accuracy: 98.39%.
Epoch [6/100], Validation Loss Decreased: 0.025919 ---> 0.002841, Model Saved.
Epoch [7/100], Loss: 0.0252, Validation: 0.00014705160950742116, Validation Accuracy: 98.37%.
Epoch [8/100], Loss: 0.0261, Validation: 4.484583680495256e-06, Validation Accuracy: 98.72%.
Epoch [8/100], Validation 

# Przetestowanie wytrenowanego modelu na danych testowych. Model osiągnął skuteczność (Accuracy) na poziomie 95.7% co jest dość dobrym wynikiem.

In [10]:
data_test = 'GermanTrafficSign/' + df_test['Path']
labels_test = df_test['ClassId']

test = CustomDataset(data_test, labels_test, transform=transform)
testloader = DataLoader(test, batch_size=32, shuffle=True)

model = GermanTrafficSignCNN().to(device)
model.load_state_dict(torch.load('CNN_GermanTrafficSign_best_model.pth'))

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in testloader:
        if torch.cuda.is_available():
            images, labels = images.to(device), labels.to(device)
        
        target = model(images)
        _, predicted = torch.max(target.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Test Accuracy: {(100 * correct / total):.2f}%.')

Test Accuracy: 95.70%.
